In [92]:
# 라이브러리 임포트
import pandas as pd
import numpy as np
# 학습용, 테스트용 데이터 분리하는 패키지
from sklearn.model_selection import train_test_split
# 스케일 조절용 패키지
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# 분류 알고리즘 사용
from sklearn.tree import DecisionTreeClassifier
# 모델 정확도 평가
from sklearn.metrics import accuracy_score
# 회귀 알고리즘
# from sklearn.neighbors import KNeighborsRegressor
# 테스트결과와 정답지 비교해서 정확도 알려줌
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score

In [ ]:
영문 컬럼명,한글 번역,의미 상세
pclass,티켓 등급,"1등석, 2등석, 3등석 (사회 경제적 지위)"
name,이름,"승객의 이름과 호칭(Mr, Miss 등) 포함"
sex,성별,"male(남성), female(여성)"
age,나이,승객의 나이 (결측치 263개)
sibsp,형제/배우자,"함께 탑승한 형제, 자매 또는 배우자의 수"
parch,부모/자녀,함께 탑승한 부모 또는 자녀의 수
ticket,티켓 번호,티켓 고유 번호
fare,요금,티켓 가격 (결측치 1개)
cabin,객실 번호,객실 구역 및 번호 (결측치 1014개)
embarked,탑승 항구,"C(쉘부르), Q(퀸즈타운), S(사우샘프턴) (결측치 2개)"
boat,구조 보트,구조되었을 때 탔던 보트 번호 (결측치 823개)
body,유해 번호,사망자 중 시신을 수습했을 때 부여된 번호 (결측치 1188개)
home.dest,고향/목적지,승객의 고향이나 도착 예정지

In [ ]:
from sklearn.datasets import fetch_openml

# 1. 전체 데이터셋(1,309개) 불러오기
# data_id=42919가 전체 승객 데이터가 포함된 버전입니다.
titanic = fetch_openml(data_id=40945, as_frame=True, parser='auto')

X = titanic.data    # 특징 데이터
y = titanic.target  # 정답(생존 여부) 데이터

# 2. 파일로 저장하기 (주피터 노트북 실행 폴더에 생성됨)
X.to_csv('titanic_X.csv', index=False)
y.to_csv('titanic_y.csv', index=False)

print(f"데이터 로드 완료! 전체 행 개수: {len(X)}")
print("파일 저장 완료: titanic_X.csv, titanic_y.csv")

In [93]:
X = pd.read_csv('titanic_X.csv')
# 경고 뜨는것 막기위해 아래 중에 하나를 사용한다.
# 1. squeeze() 사용: 1개 열만 있는 표를 'Series(한 줄)'로 압축합니다.
# y = pd.read_csv('titanic_y.csv').squeeze()
# 2. 만약 그래도 경고가 뜬다면? (가장 확실한 방법)
y = pd.read_csv('titanic_y.csv').values.ravel()

In [94]:
X.isnull().sum()

pclass          0
name            0
sex             0
age           263
sibsp           0
parch           0
ticket          0
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64

In [95]:
X.columns

Index(['pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare',
       'cabin', 'embarked', 'boat', 'body', 'home.dest'],
      dtype='object')

### - 전처리

In [96]:
df = X.copy()
df['age']=df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
df['embarked']=df['embarked'].fillna(df['embarked'].mode()[0])
df.sex = df.sex.map({'male':0, 'female':1})
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)
std_scaler = StandardScaler()
df[['Age_scaled', 'Fare_scaled']] = std_scaler.fit_transform(df[['age', 'fare']])
mms_scaler = MinMaxScaler()
df_minmax = pd.DataFrame(mms_scaler.fit_transform(df[['age', 'fare']]), columns=['Age_scaled', 'Fare_scaled'])
df['fare'] = np.log1p(df['fare'])
bins = [0,12,18,35,60,100]
labels = ['1', '2', '3', '4', '5']
df['Age_Group'] = pd.cut(df['age'], bins=bins, labels=labels)
# 가족 수 = 형제/배우자 수 + 부모/자녀 수 + 1(본인)
df['FamilySize'] = df.sibsp + df.parch + 1

# 가족 규모에 따라 혼자 탑승했는지(IsAlone) 여부를 나타내는 파생 변수 추가 생성
df['IsAlone'] = 0
df.loc[df['FamilySize']==1, 'IsAlone'] = 1
df = df.drop(['ticket', 'name', 'cabin', 'boat', 'body', 'home.dest', 'age'], axis=1)
df

,pclass,sex,sibsp,parch,fare,embarked_Q,embarked_S,Age_scaled,Fare_scaled,Age_Group,FamilySize,IsAlone
0,1,1,0,0,5.358177,False,True,-0.039005,3.442584,3,1,1
1,1,0,1,2,5.027492,False,True,-2.215952,2.286639,1,4,0
2,1,1,1,2,5.027492,False,True,-2.131977,2.286639,1,4,0
3,1,0,1,2,5.027492,False,True,0.038512,2.286639,3,4,0
4,1,1,1,2,5.027492,False,True,-0.349075,2.286639,3,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1304,3,1,1,0,2.737881,False,False,-1.163009,-0.364003,2,2,0
1305,3,1,1,0,2.737881,False,False,-0.116523,-0.364003,3,2,0
1306,3,0,0,0,2.107178,False,False,-0.232799,-0.503774,3,1,1
1307,3,0,0,0,2.107178,False,False,-0.194040,-0.503774,3,1,1


### - DecisionTreeClassifier 모델

In [97]:
# 전체 데이터를 8:2 비율로 나눕니다.
X_train, X_val, y_train, y_val = train_test_split(df, y, test_size=0.2, random_state=42)

# 1. 모델 객체 생성
model = DecisionTreeClassifier(max_depth=15, random_state=42)

# 2. 모델 학습
model.fit(X_train, y_train)

# 3. 예측 및 평가
y_pred = model.predict(X_val)
print(f"모델 정확도: {accuracy_score(y_val, y_pred) * 100:.2f}%")

모델 정확도: 79.77%


### - RandomForestClassifier 모델

In [98]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import KFold, cross_val_score

X_train, X_test, y_train, y_test = train_test_split(df, y.values.ravel(), test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(
    n_estimators=150,      # 나무 개수를 좀 더 늘림
    max_depth=10,           # 5는 너무 낮음. 8~10 정도로 상향
    # min_samples_split=5,
    min_samples_leaf=4,    # 너무 크게 잡으면 학습을 방해함
    random_state=42
)
rf_model.fit(X_train, y_train)

# 5. 성능 확인
train_score = rf_model.score(X_train, y_train)
test_score = rf_model.score(X_test, y_test)

print(f"훈련 세트 정확도: {train_score:.4f}")
print(f"테스트 세트 정확도: {test_score:.4f}")

# 교차 검증 점수 확인 (이게 진짜 실력입니다)
cv_scores = cross_val_score(rf_model, df, y.values.ravel(), cv=10)
print(f"5-Fold 교차 검증 평균: {cv_scores.mean():.4f}")

훈련 세트 정확도: 0.8720
테스트 세트 정확도: 0.7863
5-Fold 교차 검증 평균: 0.7769


### 랜덤 포레스트와 그래디언트 부스팅을 결합한 소프트 보팅(Soft Voting) 기반의 앙상블 모델

In [99]:
X_final = df.copy()
# y = y.astype(int)
rf = RandomForestClassifier(n_estimators=100, max_depth=7, min_samples_leaf=3, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)

ensemble = VotingClassifier(estimators=[('rf', rf), ('gb', gb)], voting='soft')

# [9단계] 셔플(Shuffle)을 포함한 교차 검증
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(ensemble, X_final, y.values.ravel(), cv=kf)

print(f"🚀 최종 과외 결과 - 5-Fold CV 평균: {cv_scores.mean():.4f}")
print(f"📊 각 폴드별 점수: {cv_scores}")

🚀 최종 과외 결과 - 5-Fold CV 평균: 0.8120
📊 각 폴드별 점수: [0.77862595 0.82824427 0.81679389 0.87022901 0.76628352]
